In [64]:
import requests
import json
import os
from dotenv import load_dotenv

load_dotenv(os.path.join(os.path.dirname(os.path.abspath("")), '..', '..', '..', '..', '.env'))

False

In [65]:
GRAPH_URL = "https://graph.microsoft.com/v1.0"
TOKEN_FILE = "token.json"
CLIENT_ID = os.getenv('CLIENT_ID', '').strip()
CLIENT_SECRET = os.getenv('CLIENT_SECRET', '').strip()
TENANT_ID = os.getenv('TENANT_ID', '').strip()
TOKEN_URL = f"https://login.microsoftonline.com/{TENANT_ID}/oauth2/v2.0/token"

In [66]:
def _refresh_access_token(refresh_token):
    """Use refresh token to get a new access token."""
    r = requests.post(TOKEN_URL, data={
        "client_id": CLIENT_ID,
        "client_secret": CLIENT_SECRET,
        "refresh_token": refresh_token,
        "grant_type": "refresh_token",
        "scope": "User.Read Mail.Read Mail.Send offline_access",
    })
    if not r.ok:
        print("Token refresh error:", r.json())
    r.raise_for_status()
    return r.json()


def authenticate():
    """Load access token from token.json. Auto-refresh if expired."""
    if not os.path.exists(TOKEN_FILE):
        raise Exception("token.json not found. Run fast_server_login.py and login first.")

    with open(TOKEN_FILE, "r") as f:
        token_data = json.load(f)

    access_token = token_data.get("access_token")
    if not access_token:
        raise Exception("No access_token in token.json. Re-login via fast_server_login.py.")

    # Verify token with a mail endpoint (doesn't need User.Read scope)
    headers = {"Authorization": f"Bearer {access_token}"}
    r = requests.get(f"{GRAPH_URL}/me/messages", headers=headers, params={"$top": 1})

    if r.status_code == 401:
        # Token expired — try refresh
        refresh_token = token_data.get("refresh_token")
        if not refresh_token:
            raise Exception("Token expired and no refresh_token. Re-login via fast_server_login.py.")

        print("Access token expired, refreshing...")
        new_tokens = _refresh_access_token(refresh_token)
        access_token = new_tokens["access_token"]

        # Update token.json with new tokens
        token_data["access_token"] = access_token
        if "refresh_token" in new_tokens:
            token_data["refresh_token"] = new_tokens["refresh_token"]
        token_data["expires_in"] = new_tokens.get("expires_in")
        with open(TOKEN_FILE, "w") as f:
            json.dump(token_data, f, indent=2)

        print("Token refreshed and saved.")

    print("Authenticated successfully.")
    return access_token

In [67]:
def _headers(token):
    return {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}


def get_message(token, message_id):
    """Get a single email by its Graph message ID."""
    r = requests.get(f"{GRAPH_URL}/me/messages/{message_id}", headers=_headers(token))
    r.raise_for_status()
    msg = r.json()

    email_info = {
        "id": msg["id"],
        "conversationId": msg.get("conversationId"),
        "subject": msg.get("subject"),
        "from": msg.get("from", {}).get("emailAddress", {}).get("address"),
        "to": [rec["emailAddress"]["address"] for rec in msg.get("toRecipients", [])],
        "cc": [rec["emailAddress"]["address"] for rec in msg.get("ccRecipients", [])],
        "date": msg.get("receivedDateTime"),
        "snippet": msg.get("bodyPreview"),
        "importance": msg.get("importance"),
        "isRead": msg.get("isRead"),
        "flag": msg.get("flag", {}).get("flagStatus"),
        "body_text": msg.get("body", {}).get("content") if msg.get("body", {}).get("contentType") == "text" else None,
        "body_html": msg.get("body", {}).get("content") if msg.get("body", {}).get("contentType") == "html" else None,
        "internetMessageId": msg.get("internetMessageId"),
    }
    return email_info

In [68]:
def read_emails(token, max_results=10):
    """List emails from inbox."""
    r = requests.get(
        f"{GRAPH_URL}/me/messages",
        headers=_headers(token),
        params={"$top": max_results, "$orderby": "receivedDateTime desc"},
    )
    r.raise_for_status()
    messages = r.json().get("value", [])
    return [get_message(token, msg["id"]) for msg in messages]

In [69]:
def send_email(token, to, subject, body):
    """Send a new email."""
    payload = {
        "message": {
            "subject": subject,
            "body": {"contentType": "Text", "content": body},
            "toRecipients": [{"emailAddress": {"address": to}}],
        }
    }
    r = requests.post(f"{GRAPH_URL}/me/sendMail", headers=_headers(token), json=payload)
    r.raise_for_status()

    # Fetch the sent message from SentItems to get its ID
    sent = requests.get(
        f"{GRAPH_URL}/me/mailFolders/SentItems/messages",
        headers=_headers(token),
        params={"$top": 1, "$orderby": "sentDateTime desc"},
    )
    sent.raise_for_status()
    sent_msg = sent.json().get("value", [{}])[0]

    return {
        "id": sent_msg.get("id"),
        "conversationId": sent_msg.get("conversationId"),
        "internetMessageId": sent_msg.get("internetMessageId"),
    }

In [70]:
def reply_email(token, message_id, reply_text):
    """Reply to an email by its Graph message ID."""
    payload = {
        "comment": reply_text,
    }
    r = requests.post(
        f"{GRAPH_URL}/me/messages/{message_id}/reply",
        headers=_headers(token),
        json=payload,
    )
    r.raise_for_status()

    # Fetch the latest sent message to return its details
    sent = requests.get(
        f"{GRAPH_URL}/me/mailFolders/SentItems/messages",
        headers=_headers(token),
        params={"$top": 1, "$orderby": "sentDateTime desc"},
    )
    sent.raise_for_status()
    sent_msg = sent.json().get("value", [{}])[0]

    return {
        "id": sent_msg.get("id"),
        "conversationId": sent_msg.get("conversationId"),
        "internetMessageId": sent_msg.get("internetMessageId"),
    }

In [71]:
def mark_as_important(token, message_id):
    r = requests.patch(
        f"{GRAPH_URL}/me/messages/{message_id}",
        headers=_headers(token),
        json={"importance": "high"},
    )
    r.raise_for_status()


def mark_as_star(token, message_id):
    r = requests.patch(
        f"{GRAPH_URL}/me/messages/{message_id}",
        headers=_headers(token),
        json={"flag": {"flagStatus": "flagged"}},
    )
    r.raise_for_status()


def mark_spam(token, message_id):
    """Move message to Junk Email folder."""
    r = requests.post(
        f"{GRAPH_URL}/me/messages/{message_id}/move",
        headers=_headers(token),
        json={"destinationId": "junkemail"},
    )
    r.raise_for_status()

In [72]:
def unmark_as_important(token, message_id):
    r = requests.patch(
        f"{GRAPH_URL}/me/messages/{message_id}",
        headers=_headers(token),
        json={"importance": "normal"},
    )
    r.raise_for_status()


def unstar_email(token, message_id):
    r = requests.patch(
        f"{GRAPH_URL}/me/messages/{message_id}",
        headers=_headers(token),
        json={"flag": {"flagStatus": "notFlagged"}},
    )
    r.raise_for_status()


def unspam_email(token, message_id):
    """Move message back to Inbox from Junk."""
    r = requests.post(
        f"{GRAPH_URL}/me/messages/{message_id}/move",
        headers=_headers(token),
        json={"destinationId": "inbox"},
    )
    r.raise_for_status()

In [73]:
token = authenticate()

Authenticated successfully.


In [74]:
mails = read_emails(token)
mails

[{'id': 'AQMkADAwATMwMAItNTljNy1kNjQwLTAwAi0wMAoARgAAA4bgrOu-faZBmjcKfbyInhsHAOuV-E_bFs1LrdQXTVHevRUAAAIBDAAAAOuV-E_bFs1LrdQXTVHevRUAAt6PJbgAAAA=',
  'conversationId': 'AQQkADAwATMwMAItNTljNy1kNjQwLTAwAi0wMAoAEAAUZVCC2I8CSbDHcau3MQrq',
  'subject': 'New app(s) connected to your Microsoft account',
  'from': 'account-security-noreply@accountprotection.microsoft.com',
  'to': ['saivarunchowdary4248@outlook.com'],
  'cc': [],
  'date': '2026-03-05T07:38:40Z',
  'snippet': "Microsoft account\r\nNew app(s) have access to your data\r\napexconnect connected to the Microsoft account sa**8@outlook.com.\r\nIf you didn't grant this access, please remove the app(s) from your account.\r\nManage your apps\r\nYou can also opt out or change where",
  'importance': 'normal',
  'isRead': False,
  'flag': 'notFlagged',
  'body_text': None,
  'body_html': '<html dir="ltr"><head>\r\n<meta http-equiv="Content-Type" content="text/html; charset=utf-8"><style type="text/css">\r\n<!--\r\n-->\r\n</style></head><